In [0]:
%python
# Databricks notebook source
from datetime import datetime
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

pipeline_name = "pipeline_market_prices"
execution_id = str(uuid.uuid4())
target_table = "market_prices_db.bronze.raw_food_prices"
timestamp = datetime.now()

try:
    # Caminho do volume onde o CSV foi carregado
    file_path = (
        "/Volumes/market_prices_db/bronze/landing_zone/wfp_market_food_prices.csv"
    )

    # Esquema oficial ajustado para refletir a estrutura real do WFP Market Food Prices
    explicit_schema = StructType([
        StructField("adm0_id", IntegerType(), True),
        StructField("adm0_name", StringType(), True),
        StructField("adm1_id", IntegerType(), True),
        StructField("adm1_name", StringType(), True),
        StructField("mkt_id", IntegerType(), True),
        StructField("mkt_name", StringType(), True),
        StructField("cm_id", IntegerType(), True),
        StructField("cm_name", StringType(), True),
        StructField("cur_id", IntegerType(), True),
        StructField("cur_name", StringType(), True),
        StructField("pt_id", IntegerType(), True),
        StructField("pt_name", StringType(), True),
        StructField("um_id", IntegerType(), True),
        StructField("um_name", StringType(), True),
        StructField("mp_month", IntegerType(), True),
        StructField("mp_year", IntegerType(), True),
        StructField("mp_price", DoubleType(), True),
        StructField("mp_commoditysource", StringType(), True)
    ])

    # Leitura do ficheiro CSV cru com o esquema rigoroso
    df_raw = spark.read.format("csv") \
        .option("header", "true") \
        .schema(explicit_schema) \
        .load(file_path)

    # Adição de metadados de ingestão essenciais
    df_bronze = df_raw.withColumn("ingestion_timestamp", F.lit(timestamp)) \
        .withColumn("pipeline_name", F.lit(pipeline_name)) \
        .withColumn("execution_id", F.lit(execution_id))

    # Gravação na camada Bronze usando formato Delta (com overwriteSchema para prevenir erros)
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    total_registros = df_bronze.count()
    print(f"Ingestão da camada Bronze concluída com sucesso! Total de registos: {total_registros}")

except Exception as e:
    print(f"Erro ao executar a ingestão da camada Bronze: {e}")
    raise e